
<img src=https://www.nersc.gov/_resources/themes/nersc/images/NERSC_logo_no_spacing.svg width="500">

### National Energy Research Scientific Computing Center
#### Introduction to High Performance Computing Bootcamp 2025

# Parallel Computing

**Parallel computing** is a method of computation in which many calculations or processes are carried out simultaneously.  
Instead of solving a problem sequentially on a single processor, we **divide the work** into smaller, independent (or partially independent) tasks that can run at the same time on multiple processors.  

These processors might:
- Share the same memory space (shared-memory systems)
- Communicate via a high-speed interconnect (distributed-memory systems)
- Use a hybrid model combining both approaches

The goal is to **reduce total execution time**, **solve larger problems**, or **achieve more accurate results** within the same time window.

In high-performance computing (HPC), parallel computing is critical because:
- Modern supercomputers contain **thousands to millions of processing elements**.
- Many scientific problems—like weather forecasting, molecular modeling, or climate simulations—are simply too large to be computed in a reasonable time using serial methods.
- Efficient parallelization directly impacts both performance and energy efficiency.

### Vocabulary

- **Monte Carlo method** — A computational method that uses repeated random sampling to estimate a numerical result.
- **Random sample** — One randomly generated point used as part of the estimate.
- **Independent task** — A task that can be completed without needing the result of another task first.
- **Parallelizable task** — Work that can be divided so that multiple tasks can be performed at the same time.
- **Serial execution** — Performing work one step at a time on a single process.
- **Estimate** — An approximate numerical result based on the samples collected.

---

## Part 1: Monte Carlo π Example

One simple but illustrative application of parallel computing is estimating the mathematical constant **π** using the **Monte Carlo method**.

### Part 1.1: Concept
The Monte Carlo method uses **random sampling** to solve numerical problems.  
In this case, we:
1. Generate random points inside a square of side length \( 2r \).
2. Count how many points fall inside the quarter of a circle of radius \( r \) that fits within that square.
3. Use geometry to relate the fraction of points inside the quarter circle to the value of π.

**The ratio of the areas is:**

$$
\frac{N_{\mathrm{in}}}{N_{\mathrm{total}}}
=\frac{\pi r^2}{4r^2}
=\frac{\pi}{4}
$$

**Rearranging gives:**

$$
\pi \approx \frac{4\,N_{\mathrm{in}}}{N_{\mathrm{total}}}
$$


---

### Why Parallelize It?
The Monte Carlo method requires **a large number of random samples** for good accuracy.  
If we split the work across multiple processors:
- Each processor generates its own set of random points and counts how many land inside the circle.
- The counts from all processors are **combined** to compute the final π estimate.
- The workload is perfectly parallelizable because each processor’s calculation is **independent**—no communication is required until the final sum.

This makes it a great example for:
- Learning **MPI (Message Passing Interface)** basics.
- Understanding how **data parallelism** works in HPC.
- Demonstrating how scaling up the number of processors can reduce computation time while maintaining accuracy.

---

### Visual Intuition
![PI](https://www.101computing.net/wp/wp-content/uploads/estimating-pi-monte-carlo-method.png)

In the image above:
- The square represents the total sampling area.
- The quarter circle (red curve) is the target region for counting points.
- The more random points we generate, the closer our estimate of π will get to the actual value.

---

### DOE HPC Bootcamp Context
At the DOE HPC Bootcamp, this π example is not about pushing the limits of computational mathematics—it’s about providing a **safe, simple playground** for learning parallel programming concepts.  
The Monte Carlo π problem:
- Has a **clear, visual outcome** that makes it easy to validate results.
- Allows you to **focus on MPI communication patterns** without getting lost in complex scientific code.
- Can be run on **any scale**—from your laptop to a full HPC cluster—making it ideal for demonstrating how problem size and processor count affect runtime and scaling.
  
By mastering the fundamentals here, you’ll be better prepared to apply the **same parallel computing strategies** to DOE-scale workloads like **molecular dynamics simulations, computational fluid dynamics (CFD), and climate modeling**, where parallel efficiency directly impacts both scientific discovery and operational costs.

---


### Part 1.2: Visualizing π – The Monte Carlo Way (a.k.a. Throwing Darts for Science)

Lets **watch π come to life**.

---
### What You’ll Learn


In this section, you’ll use a visualization to see how the Monte Carlo method estimates π.

As you work through the example, pay attention to:

- How each random point acts like an independent “dart throw”
- Why the estimate changes more dramatically when only a few points have been sampled
- How the estimate becomes more stable as the number of samples increases
- Why this kind of problem is a good candidate for parallel computing
---

You’ve read about the Monte Carlo method for estimating π, but there’s something magical about *seeing* the process unfold.  
In this interactive visualization, we’ll simulate the act of throwing darts at a square dartboard with a quarter circle inside.  
Every dart that lands **inside** the circle gets a red dot.  
Every dart that lands **outside** the circle gets a blue dot.  

As the simulation runs:
- The **red-to-blue ratio** starts off wobbly and uncertain.
- But as we throw more and more darts, the estimate of π begins to **settle down** toward the real value (3.1415…).
- You’ll see the estimate update live, along with a count of how many darts hit inside vs total thrown.

Why are we doing this in an HPC Bootcamp?
- It’s a **perfect parallel computing candidate**—every dart throw is independent and could be done on a separate processor.
- The logic is simple enough that you can focus on the *parallelization*, not the math.
- It’s a great way to see how randomness and statistics converge to produce accurate results over time.

Think of it as:
> *"Parallel computing meets carnival game—with the prize being π itself."*

By the end, you’ll have:
- A visual intuition for how Monte Carlo estimation works.
- A foundation for splitting the workload across multiple processors with MPI.
- The satisfaction of having calculated π without ever touching a trigonometric function.

Pro tip: The more darts you throw, the better your aim… statistically speaking. 🎯

---


### Before You Run

Before starting the visualization, think about what you expect to happen:

- Does one dart throw depend on any of the previous throws?
- When only a small number of darts have been thrown, do you expect the estimate of π to be stable or highly variable?
- What should happen to the estimate as more and more darts are added?
- Which part of this process could be performed independently by multiple processors?

In [ ]:
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import random
import time
fig, ax = plt.subplots()
#ax = fig.add_subplot(111)
circle = plt.Circle(( 0. , 0. ), 0.5 )
plt.xlim(-0.5, 0.5)
plt.ylim(-0.5, 0.5)
ax.add_patch(circle)
ax.set_aspect('equal')
N = 500
Nin = 0
t0 = time.time()
for i in range(1, N+1):
    x = random.uniform(-0.5, 0.5)
    y = random.uniform(-0.5, 0.5)
    if (np.sqrt(x*x + y*y) < 0.5):
        Nin += 1
        plt.plot([x], [y], 'o', color='r', markersize=3)
    else:
        plt.plot([x], [y], 'o', color='b', markersize=3)
    display(fig)
    plt.xlabel("$\pi$ = %3.4f \n N_in / N_total = %5d/%5d" %(Nin*4.0/i, Nin, i))
    clear_output(wait=True)

res = np.array(Nin, dtype='d')
t1 = time.time()
print(f"Pi = {res/float(N/4.0)}")
print("Time: %s" %(t1 - t0))

---
---
## Part 2: MPI Pi example

---
### What You’ll Learn

In this section, you’ll run the π calculation using multiple MPI processes.

Instead of having one process perform all of the work, MPI allows the calculation to be divided across several processes that work independently and then combine their results.

As you work through this section, pay attention to:

- How MPI identifies each process using a **rank**
- How each process determines which part of the work it should perform
- Which parts of the program run independently
- Where the separate results are combined
- How changing the number of MPI tasks affects the way the work is distributed

### Vocabulary

- **MPI (Message Passing Interface)** — A standard for communication between processes in a parallel program.
- **Process** — One running instance of a program.
- **Rank** — The unique numerical identifier assigned to an MPI process.
- **Size** — The total number of MPI processes in a communicator.
- **Communicator** — A group of MPI processes that can communicate with one another.
- **Collective operation** — An MPI operation involving all processes in a communicator.
- **Reduction** — A collective operation that combines values from multiple processes into a single result.
- **Slurm** — The workload manager used to request resources and run jobs on Perlmutter.
- **Task** — A unit of work launched by Slurm; in this example, each task corresponds to an MPI process.
- **Compute node** — A machine within the supercomputer where computational jobs run.
- **Wall time** — The maximum amount of clock time requested for a job.
---

### Part 2.1: Python example

Before implementing our method of Darts examples, lets work through a python example of usign MPI to calculate pi.

Below, you will find:

1) myPi.py - python implementation
2) myPi_jobscript(1-3).sh for submitting a batch submission on perlmutter
3) submit and stream subprocess for displaying batch submission results
4) submission block for submitting a batch job

### Before You Run

Before running the MPI program, consider the following:

- How many MPI processes do you expect to run?
- What role do `rank` and `size` play in dividing the work?
- Does each process calculate its own complete estimate of π, or only part of the final result?
- Where do the separate results need to be combined?

In [ ]:
# %%writefile myPi.py
# ^ Uncomment the line above and re-run this cell to write the code below out to
#   myPi.py, so it can be launched across many ranks with srun / mpiexec.
#   Inside the notebook the code still runs, but as a single rank (comm.size == 1).

# example of python implementation of Pi with mpi4py
from mpi4py import MPI
import numpy as np
import random
import time

# COMM_WORLD is the default communicator: the group containing every rank (process)
# in this job. comm.rank is this process's id (0, 1, 2, ...) and comm.size is how
# many there are in total. Both come from the launcher, not from the code, so the
# same script runs unchanged on 1 rank or 64.
comm = MPI.COMM_WORLD

# Method of darts: throw N random points into the square [-0.5, 0.5] x [-0.5, 0.5]
# (area 1) and count how many land inside the inscribed circle of radius 0.5
# (area pi * 0.5^2 = pi/4). The fraction inside therefore approaches pi/4,
# so pi is approximately 4 * Nin / N.
N = 5000000   # total number of darts, summed over all ranks
Nin = 0       # darts this rank found inside the circle

t0 = time.time()

# Strided loop: rank 0 takes i = 0, size, 2*size, ...; rank 1 takes 1, size+1, ...
# Every index in range(N) is handled by exactly one rank, so the N darts are split
# evenly with no communication and no overlap. This is the parallel part: each rank
# does roughly N/size iterations, so the loop should get shorter as ranks are added.
for i in range(comm.rank, N, comm.size):
    x = random.uniform(-0.5, 0.5)
    y = random.uniform(-0.5, 0.5)
    if (np.sqrt(x*x + y*y) < 0.5):   # inside the circle of radius 0.5?
        Nin += 1

# Each rank now holds only its own partial count, so the counts must be combined.
# The uppercase (buffer-based) MPI calls need numpy arrays rather than Python ints,
# hence wrapping the counts here. dtype='d' is a C double, which MPI can send directly.
res = np.array(Nin, dtype='d')       # send buffer: this rank's partial count
res_tot = np.array(Nin, dtype='d')   # receive buffer: overwritten by the call below

# Allreduce: add up res across all ranks and give the total to every rank.
# ("Reduce" would leave the answer on one rank only; the "All" prefix broadcasts it
# back to everyone.) This is the one communication step in the program.
comm.Allreduce(res, res_tot, op=MPI.SUM)

t1 = time.time()

# Every rank has the answer, but only rank 0 prints, otherwise the output would be
# repeated once per rank.
if comm.rank==0:
    # res_tot / (N/4) is the same as 4 * Nin_total / N, i.e. the estimate of pi.
    print(res_tot/float(N/4.0))
    print("Time: %s" %(t1 - t0))


How faster/slower was using the MPI-based python script than the previous one above? Check the time difference in the two code blocks.

---

### Part 2.2: Using Slurm to Control MPI Tasks

### What to Notice

The next few cells create versions of the same Slurm job script using different numbers of MPI tasks to run the same Python script as above.

As you move through them, watch the value of:

```bash
#SBATCH --ntasks-per-node=

In [ ]:
%%writefile myPi_jobscript_rank-1.sh
#!/bin/bash
#SBATCH --qos=debug
#SBATCH --time=2
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH -C cpu 
#SBATCH -A m4388

srun python myPi.py

In [ ]:
%%writefile myPi_jobscript_rank-2.sh
#!/bin/bash
#SBATCH --qos=debug
#SBATCH --time=2
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=2
#SBATCH -C cpu 
#SBATCH -A m4388

srun python myPi.py

In [ ]:
%%writefile myPi_jobscript_rank-4.sh
#!/bin/bash
#SBATCH --qos=debug
#SBATCH --time=2
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=4
#SBATCH -C cpu 
#SBATCH -A m4388

srun python myPi.py

### Submitting a Job on Perlmutter
To submit the job, run the following command:
"sbatch myPi_jobscript_rank-*.sh"

**Or** you may use the `submit_and_stream` helper method (code below), which will display the output in the notebook.

However, use only one method (direct submission or the helper script) since using both will submit identical jobs to the scheduler. Not harmful, just redundant.

In [ ]:
import subprocess, time, os, sys

def submit_and_stream(script_path: str, poll_seconds: float = 1.0):
    """
    Submit a Slurm batch script with sbatch, stream stdout to this notebook
    as the slurm-<jobid>.out file grows, and print a brief completion summary.
    """
    # Submit
    # Plain `sbatch script.sh` returns as soon as the job is queued and prints
    # "Submitted batch job 12345". Everything below is what turns that
    # fire-and-forget call into something that waits and shows you the output.
    submit_out = subprocess.check_output(["sbatch", script_path]).decode().strip()
    jobid = submit_out.split()[-1]   # the job number is the last word of that message
    print(f"Submitted job {jobid}\n")

    # Slurm writes the job's stdout/stderr to this file in the submit directory.
    # It will not exist until the job actually starts running (it may sit in the
    # queue first), so every read below is guarded by an existence check.
    outfile = f"slurm-{jobid}.out"
    last_size = 0   # how many bytes of the file we have already printed

    # Poll until the job leaves the queue, printing whatever is new each time.
    while True:
        if os.path.exists(outfile):
            with open(outfile, "r") as f:
                # Skip to where we stopped last time so nothing is printed twice,
                # then print only the bytes that have appeared since.
                f.seek(last_size)
                chunk = f.read()
                if chunk:
                    sys.stdout.write(chunk)
                    sys.stdout.flush()   # force it to appear now, not at cell end
                last_size = f.tell()     # remember the new end of file

        # Is the job still queued or running? `squeue -h -j <id>` prints one line
        # per matching job with no header, so empty output means it has finished
        # (or failed) and left the queue.
        try:
            still_running = bool(
                subprocess.check_output(["squeue", "-h", "-j", jobid]).decode().strip()
            )
        except Exception:
            # squeue can fail transiently (busy scheduler, job aged out of the
            # queue). Assume the job is still alive rather than exiting early on
            # a hiccup; the loop will re-check on the next pass.
            still_running = True

        if not still_running:
            # The job is gone, but Slurm may have flushed its last few lines after
            # our final read above, so drain whatever remains before breaking out.
            if os.path.exists(outfile):
                with open(outfile, "r") as f:
                    f.seek(last_size)
                    rest = f.read()
                    if rest:
                        sys.stdout.write(rest)
                        sys.stdout.flush()
            break

        # Wait before polling again. Checking too often just hammers the
        # scheduler; 1 second is plenty for a job of this length.
        time.sleep(poll_seconds)

    # `squeue` only knows about active jobs, so ask the accounting database how
    # the job actually ended: COMPLETED vs FAILED/TIMEOUT/CANCELLED, plus the
    # exit code. Wrapped in try/except because sacct records can lag by a few
    # seconds, and a missing summary should not look like an error.
    try:
        summary = subprocess.check_output(
            ["sacct", "-j", jobid, "--format=JobID,State,ExitCode", "-n", "-P"]
        ).decode().strip()
        print("\n--- Job summary ---")
        print(summary)
    except Exception:
        pass


In [ ]:
submit_and_stream("myPi_jobscript_rank-1.sh", poll_seconds=1.0)

In [ ]:
# or use sbatch for direct submission, uncomment the line below and run cell
#!sbatch myPi_jobscript_rank-1.sh

**Submit the rank-2 and rank-4 scripts similarly and check their outputs!**

---
---
## Part 3: Parallel computing in AI

The parallel computing in AI is usually called distributed training. Distributed training is the process of training I models across multiple GPUs or other accelerators, with the goal of speeding up the training process and enabling the training of larger models on larger datasets.

There are two ways of parallelization in distributed training. 
* **Data parallelism**: 
    * Each worker (GPU) has a complete set of model
    * different workers work on different subsets of data. 
* **Model parallelism** 
    * The model is splitted into different parts and stored on different workers
    * Different workers work on computation involved in different parts of the model
![Perlmutter](https://docs.nersc.gov/systems/perlmutter/images/PerlmutterCabinetsFinal.jpg)

### Vocabulary

- **Data parallelism** — Dividing data across multiple workers that perform similar computations on different portions of the data.
- **Distributed training** — Training a model using computational resources distributed across multiple processes or devices.
- **Worker** — A process or device responsible for performing part of a distributed computation.
- **GPU (Graphics Processing Unit)** — A processor designed to perform many computations in parallel and commonly used to accelerate AI workloads.

---
---
## Part 4: Estimating π with the "Method of Darts"

---
### What You’ll Learn

In this section, you'll put the parallel programming concepts from the lecture into practice by estimating π with the **Method of Darts**.

You'll see how the **PCAM** framework translates into an MPI program:

- **Partition:** Divide the dart throws into independent work
- **Communication:** Combine results from the MPI processes
- **Agglomeration:** Assign many dart throws to each process
- **Mapping:** Distribute the work across the available MPI processes

As you work through the example, pay attention to how each process performs its own portion of the calculation before the results are combined to produce the final estimate of π.

### Vocabulary

- **PCAM** — A framework for designing parallel algorithms using four stages: Partition, Communication, Agglomeration, and Mapping.
- **Partition** — Breaking a problem into smaller tasks that have the potential to run in parallel.
- **Communication** — Exchanging or combining information between parallel tasks.
- **Agglomeration** — Combining fine-grained tasks into larger tasks to reduce communication or other overhead.
- **Mapping** — Assigning tasks to the available compute processes.
- **Root rank** — The designated MPI process that receives the combined result of a reduction; in this example, rank 0.
- **Load balancing** — Distributing work so that processes receive approximately equal amounts of work.



---

### Part 4.1: Concept Diagram

The **method of darts** is a Monte Carlo simulation that estimates π by randomly "throwing darts" at a square target that contains a quarter circle.  

If you throw darts uniformly at random:

- The fraction that land **inside** the quarter circle is proportional to its area.
- Since the area of a quarter circle of radius 1 is π/4,  

$$\pi \approx 4 \times \frac{\text{\# darts inside}}{\text{total darts}}$$

<img src="https://upload.wikimedia.org/wikipedia/commons/8/84/Pi_30K.gif" 
     alt="Monte Carlo darts method animation" width="400">

*Above: simulation showing points inside (red) and outside (blue) the quarter circle.*

---

### Part 4.2: Parallel Implementation with MPI

In our MPI version:

1. **Partition:** Each MPI rank throws a subset of the total darts.
2. **Computation:** Each rank counts how many darts land inside the circle.
3. **Communication:** The counts are sent to **rank 0** using `MPI.Reduce`.
4. **Aggregation:** Rank 0 sums the counts and computes π.

---

### Example Run


#### Before You Run

Before running the Method of Darts program, consider the following:

- How will the total number of dart throws be divided among the MPI processes?
- Why can each process perform its assigned dart throws independently?
- What result does each process need to contribute to the final calculation?
- Where do the separate results need to be combined to produce the final estimate of π?

In [ ]:
%%writefile myPiDarts.py
# Monte Carlo "method of darts" to estimate π
# Works in serial AND with MPI (mpi4py). If you launch with mpirun/srun, it will parallelize.
# In Jupyter, this cell still runs fine with a single rank.

# `from __future__ import annotations` lets us write modern type hints such as
# `int | None` even on older Python versions, where that syntax would otherwise
# be a SyntaxError.
from __future__ import annotations
import math
import numpy as np

# Try to start up MPI. If mpi4py is missing (or no MPI library is installed),
# fall back to a single-process run instead of crashing, so the same file works
# both under `srun` and as an ordinary `python myPiDarts.py`.
try:
    from mpi4py import MPI
    COMM = MPI.COMM_WORLD    # the group of all processes in this job
    RANK = COMM.Get_rank()   # this process's id: 0, 1, 2, ... SIZE-1
    SIZE = COMM.Get_size()   # how many processes there are in total
except Exception:  # mpi4py not available; fall back to serial
    COMM = None
    RANK = 0
    SIZE = 1

def darts_inside_circle(n: int, seed: int | None = None) -> int:
    """
    Throw n 'darts' uniformly at the unit square [0,1]x[0,1] and
    count how many land inside the quarter circle x^2 + y^2 <= 1.
    Vectorized for speed.
    """
    # Make each rank reproducible but distinct
    # This matters: if every rank used the same seed, every rank would throw the
    # *identical* darts. Combining N copies of the same answer is no better than
    # one, so the error would stop shrinking as ranks are added.
    if seed is not None:
        # Use SeedSequence to derive independent per-rank seeds
        # spawn() splits one root seed into SIZE statistically independent
        # streams; this rank takes stream number RANK. The run stays fully
        # reproducible while the ranks stay independent of one another.
        ss = np.random.SeedSequence(seed)
        child_ss = ss.spawn(1 if SIZE == 1 else SIZE)[RANK]
        rng = np.random.default_rng(child_ss)
    else:
        # No seed given: numpy seeds from OS entropy, so every run differs.
        rng = np.random.default_rng()

    # Vectorized throw: generate all n x-coordinates and all n y-coordinates as
    # numpy arrays in one shot, instead of looping in Python one dart at a time.
    # This is why this version is much faster than the plain-loop myPi.py.
    x = rng.random(n)
    y = rng.random(n)
    # x*x + y*y <= 1.0 gives an array of True/False, one entry per dart;
    # count_nonzero counts the Trues, i.e. the darts inside the quarter circle.
    return np.count_nonzero(x*x + y*y <= 1.0)

def estimate_pi(n_samples: int, seed: int | None = 12345) -> tuple[float, int, int]:
    """
    Estimate π using the Monte Carlo 'darts' method.
    Each rank throws n_samples // SIZE darts; the first (n_samples % SIZE)
    ranks throw one extra, so the totals sum to exactly n_samples.
    Returns (pi_estimate, local_inside, global_inside) on rank 0;
    on other ranks, returns (np.nan, local_inside, 0) for convenience.
    """
    # Divide the work as evenly as possible across ranks
    # This is the PARTITION step. n_samples rarely divides evenly by SIZE, so the
    # first `rem` ranks each take one extra dart. That keeps the loads within one
    # dart of each other (load balancing) and makes the per-rank counts add up to
    # exactly n_samples rather than to a rounded-down total.
    base = n_samples // SIZE
    rem = n_samples % SIZE
    local_n = base + (1 if RANK < rem else 0)

    # COMPUTATION: every rank works only on its own darts, with no communication
    # at all. That independence is what makes this problem embarrassingly parallel.
    local_inside = darts_inside_circle(local_n, seed=seed)

    # COMMUNICATION: each rank holds only a partial count, so the counts must be
    # combined. Lowercase `reduce` is the general-purpose mpi4py call: it pickles
    # ordinary Python objects, so plain ints work directly (the uppercase
    # Allreduce in myPi.py needs numpy buffers instead). root=0 sends the summed
    # result to rank 0 only; every other rank receives None.
    if COMM is not None:
        global_inside = COMM.reduce(local_inside, op=MPI.SUM, root=0)
        # The dart totals are summed too, rather than assuming n_samples, so the
        # arithmetic stays right no matter how the remainder was distributed.
        global_total = COMM.reduce(local_n,      op=MPI.SUM, root=0)
    else:
        # Serial fallback: this single process already holds everything.
        global_inside = local_inside
        global_total = local_n

    # AGGREGATION: only rank 0 received the sums, so only rank 0 can finish the
    # calculation. The quarter circle occupies pi/4 of the unit square, so the
    # fraction of darts inside is an estimate of pi/4, hence the factor of 4.
    if RANK == 0:
        pi_hat = 4.0 * (global_inside / float(global_total))
        return pi_hat, local_inside, global_inside
    else:
        # Other ranks have no estimate to report; nan says so explicitly rather
        # than handing back a misleading number.
        return float("nan"), local_inside, 0

# --- Configure and run ---
# Feel free to turn this up to 10_000_000+ when running on multiple ranks
# Monte Carlo error shrinks like 1/sqrt(N), so each extra correct digit costs
# roughly 100x the darts. Adding ranks is how you afford that.
N_SAMPLES = 2_000_000
SEED = 2025

pi_hat, local_inside, global_inside = estimate_pi(N_SAMPLES, seed=SEED)

# Guard the printing with RANK == 0; otherwise every rank prints its own copy and
# the output appears SIZE times.
if RANK == 0:
    rel_err = abs(pi_hat - math.pi) / math.pi
    print(f"Ranks: {SIZE}")
    print(f"Total darts: {N_SAMPLES:,}")
    print(f"Estimated π: {pi_hat:.8f}")
    print(f"Actual    π: {math.pi:.8f}")
    print(f"Relative error: {rel_err:.3e}")


#### How This Version Differs from `myPi.py`

Both programs estimate the same number by the same Monte Carlo idea, and both are
parallelised with MPI. What changes is *how* each step is carried out, so it is worth
comparing them side by side before you run.

| | `myPi.py` (Part 2) | `myPiDarts.py` (Part 4) |
|---|---|---|
| **Geometry** | square from -0.5 to 0.5, circle of radius 0.5 | unit square from 0 to 1, quarter circle of radius 1 |
| **Throwing the darts** | Python `for` loop, one dart at a time with `random.uniform` | vectorised NumPy, all darts generated as arrays in one call |
| **Partition** | *strided*: rank r takes darts r, r+size, r+2*size, ... | *block*: each rank takes a contiguous share, first `rem` ranks get one extra |
| **Random seeds** | none set, so each process seeds itself from OS entropy | one root seed split with `SeedSequence.spawn()` into independent per-rank streams |
| **Combining results** | `comm.Allreduce` (uppercase), needs NumPy buffers, answer goes to **every** rank | `COMM.reduce` (lowercase), works on plain Python ints, answer goes to **rank 0 only** |
| **If MPI is missing** | crashes on import | falls back to a single-process serial run |
| **Reports** | the estimate and the elapsed time | rank count, dart count, estimate, true value, and relative error |

**What to notice, and why each change matters:**

1. **Vectorisation is not parallelism.** Replacing the Python loop with NumPy arrays makes
   a single rank much faster, but it does not involve MPI at all. The two speedups are
   independent, and you get both at once here. This is a common source of confusion when
   reading timing results: a faster run does not automatically mean the parallelism worked.

2. **Seeding matters in Monte Carlo.** If every rank started from the same seed, every rank
   would throw the *identical* darts. Averaging many copies of one answer is no better than
   the answer itself, so the error would stop improving as ranks were added. `spawn()` gives
   each rank an independent stream while keeping the whole run reproducible.

3. **`reduce` versus `Allreduce`.** Use `Allreduce` when every rank needs the combined result
   in order to keep working. Use `reduce` when only one rank needs it, as here, where rank 0
   simply prints the answer. `reduce` moves less data. The lowercase and uppercase spellings
   are also a real mpi4py distinction: lowercase accepts any picklable Python object,
   uppercase requires a buffer such as a NumPy array and is faster for large arrays.

4. **Strided versus block partitioning.** Both split the work evenly here because every dart
   costs the same. When the cost per item varies, the choice starts to affect load balance.

5. **Reporting the relative error** turns the run into a measurement. Monte Carlo error falls
   off like 1/sqrt(N), so watch how much (or how little) the error improves when you increase
   the number of ranks in the job script.

In [ ]:
%%writefile myPiDarts_jobscript.sh
#!/bin/bash
#SBATCH --qos=debug
#SBATCH --time=2
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=8
#SBATCH -C cpu 
#SBATCH -A m4388

srun python myPiDarts.py


### Before You Submit

Before submitting the job to Slurm, take a look at the job script:

- How many MPI tasks will this job launch?
- Which Slurm directive controls the number of tasks?
- Which command launches the Method of Darts program across those tasks?
- How do you expect the work performed by each process to change compared with a single-process run?

In [ ]:
submit_and_stream("myPiDarts_jobscript.sh", poll_seconds=1.0)